## Data Preparation

You should prepare the following before running this step. Please refer to the `example_data` folder for guidance:

1. **the motion-corrupted data**
   - We prepared two examples. First one is a simulated case ```example_data/data/simulations/00014689/0000455416```, the other is a real-world portable CT case ```example_data/data/portable_CT/00102121/0000156734```

2. **A patient list** that enumerates all your cases
   - To understand the standard format, please refer to the file:  
     `example_data/Patient_list/patient_list_simulated.xlsx` or `example_data/Patient_list/patient_list_portable.xlsx`

3. **Histogram equalization** 
   - please run ```histogram_equalization.ipynb``` to generate the bins needed for histogram equalization data preprocessing.
   - we have prepared the bins generated by our own dataset in ```example_data/histogram_equalization```, you can directly use it if needed.

---

### Docker environment
Please use `docker/docker_pytorch`, it will build a pytorch-based container


In [1]:
# import
import sys 
sys.path.append('/workspace/Documents')
import os
import torch
import numpy as np
import nibabel as nb
import Diffusion_for_CT_motion.diffusion_models.conditional_diffusion_3D as ddpm_3D
import Diffusion_for_CT_motion.diffusion_models.conditional_EDM_3D as edm
import Diffusion_for_CT_motion.functions_collection as ff
import Diffusion_for_CT_motion.utils.Build_list as Build_list
import Diffusion_for_CT_motion.utils.Generator as Generator

main_path = '/mnt/camca_NAS/diffusion_ct_motion' # replace with your main path

/usr/local/lib/python3.8/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/workspace/Documents/Diffusion_for_CT_motion/diffusion_models/conditional_diffusion_3D.py:864: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled = False)


In [2]:
def sliding_windows(N, window=20, step=10):
    result = []
    start = 0
    while start + window <= N:
        start_slice = start
        end_slice = start + window
        if end_slice <= N and start_slice+ step + window >N:
            end_slice = N
            result.append([start_slice, end_slice])
            break
        else:
            result.append([start_slice, start_slice + window])
        
        start += step
    
    return result


#### step 1: set the trained diffusion model path

In [8]:
trial_name = 'diffusion_model'
trained_model_filename = os.path.join(main_path, 'example_data/models',trial_name, 'models', 'model-final.pt') # replace with your own path
# save_folder = os.path.join(main_path,'example_data/models', trial_name, 'pred_image_simulated') 
save_folder = os.path.join(main_path,'example_data/models', trial_name, 'pred_image_portable') 
os.makedirs(save_folder, exist_ok=True)

#### step 2: set the patient list

In [13]:
# define train
# build_sheet =  Build_list.Build(os.path.join(main_path, 'example_data/Patient_list', 'patient_list_simulated.xlsx'))
build_sheet =  Build_list.Build(os.path.join(main_path, 'example_data/Patient_list', 'patient_list_portable.xlsx'))
_,patient_id_list,patient_subid_list, x0_list_test, condition_list_test = build_sheet.__build__(batch_list = [0]) 

#### step 3: set some default parameter

In [14]:
# set default, don't change unless necessary
image_size_3D = [256,256,20] 
patch_size = image_size_3D[0] # apply on whole image

# don't change the following, 
histogram_equalization = True # alreayd set True
# these two are used for histogram equalization
bins = np.load(os.path.join(main_path,'example_data/histogram_equalization/bins.npy'))
bins_mapped = np.load(os.path.join(main_path,'example_data/histogram_equalization/bins_mapped.npy'))

# for data normalization 
background_cutoff = -1000 
maximum_cutoff = 2000
normalize_factor = 'equation'

# sample steps:
num_sample_steps = 50 

#### step 4: define the U-Net and the EDM

In [15]:
# main code
model = ddpm_3D.Unet3D(
    init_dim = 64,
    channels = 1, 
    dim_mults = (1, 2, 4, 8),
    flash_attn = False,
    conditional_diffusion = True,
    full_attn = (None, None, False, True),
)


#### step 5: sample

In [16]:
for i in range(0,x0_list_test.shape[0]):
    
    x0_file = x0_list_test[i]
    condition_file = condition_list_test[i]

    patient_id = patient_id_list[i]; patient_subid = patient_subid_list[i]
 

    save_folder_case = os.path.join(save_folder, patient_id,patient_subid)
    ff.make_folder([os.path.dirname(save_folder_case), save_folder_case])

    # load the condition file
    condition_img = nb.load(condition_file).get_fdata()
    z_slice_num = condition_img.shape[2]
    slice_range_list = sliding_windows(N= z_slice_num, window=20, step=10)
    print('slice range list:', slice_range_list)

    for ii in range(0, len(slice_range_list)):
        slice_range = slice_range_list[ii]
        print('slice range:', slice_range[0], 'to', slice_range[1])
        save_file_name = os.path.join(save_folder_case, 'pred-slice' + str(slice_range[0]) +'to' + str(slice_range[1])+ '.nii.gz')

        if os.path.exists(save_file_name):
            print('file already exists, skip')
            continue
 

        diffusion_model = edm.EDM(
            model,
            image_size = [image_size_3D[0], image_size_3D[1], slice_range[1] - slice_range[0]],
            num_sample_steps = num_sample_steps,
            clip_or_not = True,
            clip_range = [-1,1],)
        
    
        generator = Generator.Dataset_dual_patch(
            np.array([condition_file]), # in this case, we don't have x0 file (the motion-free reference)
            np.array([condition_file]),

            image_size_3D = [image_size_3D[0], image_size_3D[1], slice_range[1] - slice_range[0]],
            slice_start = slice_range[0],
            slice_num = slice_range[1] - slice_range[0],

            patch_size = patch_size,
            patch_stride = 1,
            original_patch_num = 1,
            random_sampled_patch_num = 0,
              
            histogram_equalization = histogram_equalization, 
            bins = bins,
            bins_mapped = bins_mapped,
                
            background_cutoff = background_cutoff, 
            maximum_cutoff = maximum_cutoff,
            normalize_factor = normalize_factor,)

            
        # sample:
        sampler = edm.Sampler(
            diffusion_model,
            generator,
            image_size = image_size_3D,
            batch_size = 1)

        sampler.sample_3D_w_trained_model(trained_model_filename=trained_model_filename, 
                                    motion_image_file =  condition_file, 
                                    save_file = save_file_name,
                                    slice_range = slice_range)
                                        
  
    print('finish sampling')
    
    # annel the slices
    affine = nb.load(condition_file).affine
    final_image = np.zeros(condition_img.shape)

    for j in range(0, len(slice_range_list)):
        slice_range = slice_range_list[j]
        slice_image = nb.load(os.path.join(save_folder_case, 'pred-slice' + str(slice_range[0]) +'to' + str(slice_range[1])+ '.nii.gz')).get_fdata()
        
        if j == 0:
            final_image[:,:,0:15] = slice_image[:,:,0:15]
        elif j != 0 and j != len(slice_range_list) - 1:
            final_image[:,:,15 + (j-1)*10: 15 + j*10] = slice_image[:,:,5:15]
        else:
            final_image[:,:, 15+(j-1)*10 : condition_img.shape[2]] = slice_image[:,:,5:]
    nb.save(nb.Nifti1Image(final_image, affine), os.path.join(save_folder_case, 'pred.nii.gz'))

slice range list: [[0, 20], [10, 30], [20, 40], [30, 55]]
slice range: 0 to 20
model device:  cuda:0


sampling time step: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]


(256, 256, 20)
slice range: 10 to 30
model device:  cuda:0


sampling time step: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]


(256, 256, 20)
slice range: 20 to 40
model device:  cuda:0


sampling time step: 100%|██████████| 50/50 [00:53<00:00,  1.07s/it]


(256, 256, 20)
slice range: 30 to 55
model device:  cuda:0


sampling time step: 100%|██████████| 50/50 [01:14<00:00,  1.48s/it]


(256, 256, 25)
finish sampling
